# ИЗ-3. Таблицы: чтение, соединение, группировка

**Евстафьева Вера, ИТ-14-2024, вариант 06** — заполните эту строку.

Перед сдачей: **Kernel → Restart & Run All**. Работа, которая так не проходит,
не проверяется. Выводы ячеек должны остаться в файле — ноутбук с пустыми
выводами смотреть не в чем.

Файл называется `ИЗ-3_Евстафьева_Вера_вариант06.ipynb`, сдача через Moodle.


## Настройка

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

SEED = 42                      # одно зерно на всю работу
np.random.seed(SEED)
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 20)

df = pd.read_csv("data.csv")    # файл лежит рядом с ноутбуком
df.shape


(2485, 20)

## ● Пункт 1. Чтение и типы

<sub>обязательный пункт · 1 балл</sub>

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data.csv")

print("Типы столбцов после чтения (df.dtypes):")
print(df.dtypes)

converted = {}
for col in df.columns:
    before = df[col].isna().sum()
    as_num = pd.to_numeric(df[col], errors="coerce")
    after = as_num.isna().sum()
    if after > before:
        converted[col] = after - before

if converted:
    print("\nСтолбцы, которые пришлось приводить к числу:")
    for col, n in converted.items():
        print(f"  {col}: {n} новых NaN после приведения")
    for col in converted:
        df[col] = pd.to_numeric(df[col], errors="coerce")
else:
    print("\nПриводить не пришлось ни одного столбца: все столбцы уже числовые.")


Типы столбцов после чтения (df.dtypes):
row_id             int64
y_clf              int64
price            float64
bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
grade              int64
sqft_above         int64
sqft_basement    float64
yr_built           int64
yr_renovated       int64
zipcode            int64
lat              float64
long             float64
sqft_living15      int64
sqft_lot15         int64
dtype: object

Приводить не пришлось ни одного столбца: все столбцы уже числовые.


**Ответ.** _Здесь письменный ответ на вопрос пункта._

> Приводить не пришлось ни одного столбца: после pd.read_csv все 20 столбцов уже имеют числовой тип (int64 или float64). Убедится в этом помогла проверка через pd.to_numeric(..., errors="coerce"): для каждого столбца сравнивалось число NaN до и после приведения, и ни в одном столбце новых NaN не появилось. Единственный столбец с пропусками - sqft_basement (как и впрошлом задании), но эти NaN возникли из-за пустых ячеек в исходных данных, а сам столбец распознался как числовой правильно.

## ● Пункт 2. Пропуски и их механизм

<sub>обязательный пункт · 1 балл</sub>

In [3]:
import pandas as pd
import numpy as np

miss_table = pd.DataFrame({
    "Столбец": df.columns,
    "Число пропусков": df.isna().sum().values,
    "Доля пропусков": df.isna().mean().values
}).sort_values("Доля пропусков", ascending=False)

print("Таблица пропусков:")
print(miss_table[miss_table["Число пропусков"] > 0].to_string(index=False))

target_col = miss_table.iloc[0]["Столбец"]
mask_missing = df[target_col].isna()

print(f"\nЦелевой столбец: {target_col}")
print(f"Всего пропусков: {int(mask_missing.sum())} из {len(df)} ({mask_missing.mean():.2%})\n")

print("Связь с числовыми столбцами:")
for suspect in ["price", "sqft_living", "sqft_above", "yr_built", "grade"]:
    print(f"\nСтолбец '{suspect}':")

    stats = df.groupby(mask_missing)[suspect].agg(["count", "mean", "median", "std"])

    stats.index = ["Нет пропуска (есть подвал)", "Есть пропуск (нет подвала)"]
    print(stats.round(2))

print("\nСвязь с категориальными столбцами (доли категорий):")
for suspect in ["waterfront", "view", "y_clf", "floors"]:
    print(f"\nСтолбец '{suspect}':")

    shares = df.groupby(mask_missing)[suspect].value_counts(normalize=True).round(3)
    shares.index.names = ["Наличие пропуска", "Значение"]
    print(shares)

Таблица пропусков:
      Столбец  Число пропусков  Доля пропусков
sqft_basement              290          0.1167

Целевой столбец: sqft_basement
Всего пропусков: 290 из 2485 (11.67%)

Связь с числовыми столбцами:

Столбец 'price':
                            count       mean    median        std
Нет пропуска (есть подвал)   2195  508907.34  450000.0  274951.76
Есть пропуск (нет подвала)    290  486049.45  436900.0  250481.69

Столбец 'sqft_living':
                            count     mean  median     std
Нет пропуска (есть подвал)   2195  1991.29  1820.0  874.37
Есть пропуск (нет подвала)    290  1925.98  1820.0  770.64

Столбец 'sqft_above':
                            count     mean  median     std
Нет пропуска (есть подвал)   2195  1707.87  1480.0  790.24
Есть пропуск (нет подвала)    290  1598.36  1460.0  664.25

Столбец 'yr_built':
                            count     mean  median    std
Нет пропуска (есть подвал)   2195  1967.50  1969.0  29.47
Есть пропуск (нет подвала)    290

**Ответ.** _Здесь письменный ответ на вопрос пункта._

> Пропуски в sqft_basement (290 из 2485, 11.67%) не случайны - они связаны с другими признаками. У домов с пропуском (нет подвала) средняя надземная площадь sqft_above равна 1598, а у домов с подвалом - 1708; средняя sqft_living - 1926 против 1991; средний год постройки - 1960 против 1968; средний grade - 7.35 против 7.54. Особенно сильно различие в столбце y_clf: среди домов с пропуском подвала доля y_clf = 1 составляет 77.6%, а среди домов без пропуска - лишь 29.6%.

>Это означает, что пропуск - не «неизвестное значение», а маркер определённого типа домов: более старых, чуть меньших по площади и с высокой долей y_clf = 1. Если заполнить эти пропуски общим средним по столбцу sqft_basement (≈283), мы припишем подвал именно той группе старых и меньших домов, у которых его в реальности нет. Данные «поедут» вверх по площади подвала для этой подгруппы, причём сдвиг будет односторонним - в сторону более новых и больших домов, где подвал есть. Правильнее заполнить эти пропуски нулём, то есть обозначить, что подвала нет.

## ● Пункт 3. Дубликаты и невозможные значения

<sub>обязательный пункт · 1 балл</sub>

In [4]:
import pandas as pd
import numpy as np

n_full_dup = df.duplicated().sum()
print(f"Полных дубликатов (все столбцы совпадают): {n_full_dup}")

subset_cols = ["price", "sqft_living", "sqft_lot", "yr_built", "lat", "long"]
n_partial_dup = df.duplicated(subset=subset_cols).sum()
print(f"Частичных дубликатов по {subset_cols}: {n_partial_dup}")

if n_partial_dup:
    dup_rows = df[df.duplicated(subset=subset_cols, keep=False)].sort_values(subset_cols)
    print("\nПримеры строк, попавших в частичные дубликаты:")
    print(dup_rows[subset_cols + ["bedrooms", "bathrooms", "sqft_above"]].head(6).to_string(index=False))

print("\nПроверка невозможных значений:")

bad_bed = df[df["bedrooms"] < 0]
print(f"\nbedrooms < 0: {len(bad_bed)} строк")
if len(bad_bed):
    print(bad_bed[["bedrooms", "bathrooms", "sqft_living", "price"]].to_string(index=False))

bad_bath = df[df["bathrooms"] < 0]
print(f"\nbathrooms < 0: {len(bad_bath)} строк")
if len(bad_bath):
    print(bad_bath[["bedrooms", "bathrooms", "sqft_living", "price"]].to_string(index=False))

bad_price = df[df["price"] <= 0]
print(f"\nprice <= 0: {len(bad_price)} строк")
if len(bad_price):
    print(bad_price[["price", "sqft_living", "yr_built"]].to_string(index=False))

bad_sqft = df[(df["sqft_living"] <= 0) | (df["sqft_above"] <= 0) | (df["sqft_lot"] <= 0)]
print(f"\nsqft_living/sqft_above/sqft_lot <= 0: {len(bad_sqft)} строк")
if len(bad_sqft):
    print(bad_sqft[["sqft_living", "sqft_above", "sqft_lot"]].to_string(index=False))

bad_year = df[(df["yr_built"] < 1800) | (df["yr_built"] > 2026)]
print(f"\nyr_built вне [1800, 2026]: {len(bad_year)} строк")
if len(bad_year):
    print(bad_year[["yr_built", "price", "sqft_living"]].to_string(index=False))

bad_lat = df[(df["lat"] < -90) | (df["lat"] > 90)]
bad_long = df[(df["long"] < -180) | (df["long"] > 180)]
print(f"\nlat вне [-90, 90]: {len(bad_lat)} строк")
print(f"long вне [-180, 180]: {len(bad_long)} строк")


Полных дубликатов (все столбцы совпадают): 0
Частичных дубликатов по ['price', 'sqft_living', 'sqft_lot', 'yr_built', 'lat', 'long']: 1

Примеры строк, попавших в частичные дубликаты:
   price  sqft_living  sqft_lot  yr_built     lat     long  bedrooms  bathrooms  sqft_above
259950.0         1070       649      2008 47.5213 -122.357         2        2.0         720
259950.0         1070       649      2008 47.5213 -122.357         2        2.0         720

Проверка невозможных значений:

bedrooms < 0: 3 строк
 bedrooms  bathrooms  sqft_living     price
       -2       3.25         4475 1035480.0
       -2       2.50         3720  899000.0
       -2       2.25         1840  350000.0

bathrooms < 0: 0 строк

price <= 0: 0 строк

sqft_living/sqft_above/sqft_lot <= 0: 0 строк

yr_built вне [1800, 2026]: 0 строк

lat вне [-90, 90]: 0 строк
long вне [-180, 180]: 0 строк


**Ответ.** _Здесь письменный ответ на вопрос пункта._

> Найдено два типа проблем:

> Первая: один частичный дубликат по набору [price, sqft_living, sqft_lot, yr_built, lat, long]: две строки полностью совпадают по этим шести признакам, то есть это один и тот же дом, проданный дважды или ошибочно занесённый в базу. Я считаю самым логичным решением - удалить одну из этих строк.

> Вторая: три строки с bedrooms = -2. Отрицательное число спален физически невозможно, это явная ошибка ввода (скорее всего, при заполнении поставили лишний минус). Я бы заменила эти три значения на просто "2" или же пропуском (NaN), а в дальнейшем такие пропуски можно будет заполнить средним, но уже зная, что это восстановленное значение.

> Полных дубликатов, нулевых площадей, отрицательных цен и некорректных координат в данных нет.

## ● Пункт 4. Группировка с выводом

<sub>обязательный пункт · 1 балл</sub>

In [5]:
import pandas as pd
import numpy as np

group_col = "view"
value_col = "price"

summary = (
    df.groupby(group_col)[value_col]
      .agg(["count", "mean", "median", "std", "min", "max"])
      .sort_values("mean")
      .round(2)
)

summary.columns = ["Размер группы", "Среднее", "Медиана", "СКО", "Минимум", "Максимум"]
print(f"Сводка по '{group_col}' и '{value_col}':")
print(summary.to_string())

lowest = summary.index[0]
highest = summary.index[-1]
mean_low = summary.loc[lowest, "Среднее"]
mean_high = summary.loc[highest, "Среднее"]
std_low = summary.loc[lowest, "СКО"]
std_high = summary.loc[highest, "СКО"]
diff = mean_high - mean_low

print(f"\nКрайние группы: {lowest} (среднее {mean_low}) и {highest} (среднее {mean_high})")
print(f"Разница средних: {diff:.2f}")
print(f"СКО внутри групп: у {lowest} - {std_low}, у {highest} - {std_high}")
print(f"Разница / среднее СКО: {diff / ((std_low + std_high) / 2):.2f}")
print(f"Размеры крайних групп: {int(summary.loc[lowest, 'Размер группы'])} и {int(summary.loc[highest, 'Размер группы'])}")


Сводка по 'view' и 'price':
      Размер группы     Среднее    Медиана         СКО   Минимум   Максимум
view                                                                       
0              2302   482718.42   440750.0   205949.72   75000.0  1800000.0
1                13   560526.00   540000.0   181347.05  255000.0   900000.0
2                98   639665.12   563500.0   371361.58  192500.0  2725000.0
3                52   818427.50   616900.0   555296.37  265000.0  2450000.0
4                20  1712795.00  1748500.0  1020669.03  599950.0  4500000.0

Крайние группы: 0 (среднее 482718.42) и 4 (среднее 1712795.0)
Разница средних: 1230076.58
СКО внутри групп: у 0 - 205949.72, у 4 - 1020669.03
Разница / среднее СКО: 2.01
Размеры крайних групп: 2302 и 20


**Ответ.** _Здесь письменный ответ на вопрос пункта._

> Категориальный столбец view и числовой price. По сводке крайние группы различаются сильно: у view = 0 среднее 482 718, у view = 4 среднее 1 712 795, разница 1 230 077. Это в 2.01 раза больше среднего СКО внутри этих групп, то есть формально различие больше разброса. Но надёжность у групп разная: view = 0 имеет 2302 наблюдения, а view = 4 всего 20. Поэтому различие выглядит настоящим по направлению (дома с видом 4 действительно дороже), но конкретное значение разницы по группе из 20 наблюдений статистически очень ненадёжно.

## ○ Пункт 5. Сводная таблица по двум признакам

<sub>дополнительный пункт · 1 балл</sub>

In [6]:
import pandas as pd
import numpy as np

df["price_bin"] = pd.qcut(df["price"], q=4, labels=["1", "2", "3", "4"])

pivot_mean = pd.pivot_table(
    df,
    index="view",
    columns="price_bin",
    values="sqft_living",
    aggfunc="mean",
    margins=True,
    margins_name="Всего",
    observed=False
).round(1)

pivot_count = pd.pivot_table(
    df,
    index="view",
    columns="price_bin",
    values="sqft_living",
    aggfunc="count",
    margins=True,
    margins_name="Всего",
    observed=False
).fillna(0).astype(int)

print("Средняя площадь sqft_living:")
print(pivot_mean.to_string())

print("\nРазмеры групп (число наблюдений в клетке):")
print(pivot_count.to_string())

Средняя площадь sqft_living:
price_bin       1       2       3       4   Всего
view                                             
0          1320.8  1663.5  2018.6  2857.1  1932.3
1           810.0  2276.7  2395.0  2828.0  2412.3
2          1612.9  1858.9  2258.5  2826.8  2321.2
3          1944.0  1965.0  2477.3  3294.6  2736.2
4             NaN     NaN  1970.0  4115.8  4008.5
Всего      1334.8  1673.9  2039.5  2912.9  1983.7

Размеры групп (число наблюдений в клетке):
price_bin    1    2    3    4  Всего
view                                
0          595  611  579  517   2302
1            1    3    4    5     13
2           21    9   27   41     98
3            5   10   11   26     52
4            0    0    1   19     20
Всего      622  633  622  608   2485


**Ответ.** _Здесь письменный ответ на вопрос пункта._

> Самая необычная клетка: view = 4 * 3. Её средняя площадь там 1970, а рядом в 4 - 4115.8. Но в таблице размеров видно, что в этой клетке всего 1 наблюдение. То же самое в строке view = 1: там по 1–5 домов в клетке, и средние от 810 до 2828 - это шум.

> Доверять можно только строке view = 0: в ней от 517 до 595 наблюдений в клетке, и видна устойчивая закономерность - с ростом цены площадь растёт от 1320.8 до 2857.1. По view = 2 и view = 3 закономерность тоже есть, но слабее.

## ○ Пункт 6. Производный признак

<sub>дополнительный пункт · 1 балл</sub>

In [7]:
import pandas as pd
import numpy as np

df["price_per_sqft"] = df["price"] / df["sqft_living"]

df["price_per_sqft"] = df["price_per_sqft"].replace([np.inf, -np.inf], np.nan)

print("Описательная статистика:")
stats = df[["price", "price_per_sqft"]].describe().round(2)
stats.columns = ["Цена (price)", "Цена за кв. фут (price_per_sqft)"]
print(stats.to_string())

cv_price = df["price"].std() / df["price"].mean()
cv_per_sqft = df["price_per_sqft"].std() / df["price_per_sqft"].mean()
print(f"\nКоэффициент вариации цены: {cv_price:.4f}")
print(f"Коэффициент вариации цены за кв. фут: {cv_per_sqft:.4f}")

print("\nКорреляция price_per_sqft с исходными столбцами:")
for col in ["price", "sqft_living"]:
    r = df["price_per_sqft"].corr(df[col])
    print(f"price_per_sqft - {col}: r = {r:+.4f}")


Описательная статистика:
       Цена (price)  Цена за кв. фут (price_per_sqft)
count       2485.00                           2485.00
mean      506239.82                            264.86
std       272260.96                             87.76
min        75000.00                             88.08
25%       342450.00                            207.89
50%       450000.00                            254.87
75%       600000.00                            306.60
max      4500000.00                            787.04

Коэффициент вариации цены: 0.5378
Коэффициент вариации цены за кв. фут: 0.3314

Корреляция price_per_sqft с исходными столбцами:
price_per_sqft - price: r = +0.3794
price_per_sqft - sqft_living: r = -0.2529


**Ответ.** _Здесь письменный ответ на вопрос пункта._

> Признак price_per_sqft является ценой за квадратный фут. Он показывает «удельную стоимость» дома, она позволяет сравнить дома разного метража по их реальной ценности (качеству, локации). Коэффициент вариации у нового признака (0.3314) заметно ниже, чем у исходной цены (0.5378), то есть он более устойчив к выбросам. Корреляция с исходными столбцами низкая, то есть признак не дублирует их, а даёт новую информацию.

## ○ Пункт 7. Длинный хвост категорий

<sub>дополнительный пункт · 1 балл</sub>

In [8]:
import pandas as pd
import numpy as np

candidates = ["grade", "bedrooms", "bathrooms", "floors", "view", "waterfront", "y_clf"]
nunique = {col: df[col].nunique() for col in candidates}
print("Уникальных значений по кандидатам:", nunique)

cat_col = max(nunique, key=nunique.get)
print(f"\nВыбранный столбец: {cat_col}, уникальных значений: {nunique[cat_col]}")

counts = df[cat_col].value_counts()
n_total = len(df)

top10 = counts.head(10)
share_top10 = top10.sum() / n_total
print(f"Десять самых частых значений покрывают {share_top10:.2%} данных")

n_once = (counts == 1).sum()
print(f"Значений, встретившихся ровно один раз: {n_once}")

cum_share = counts.cumsum() / n_total
keep = cum_share[cum_share <= 0.90].index.tolist()

if cum_share.iloc[len(keep)] < 1.0:
    keep.append(counts.index[len(keep)])
print(f"\nКатегорий, покрывающих 90% данных: {len(keep)}")
print(f"Значения: {keep}")

df[f"{cat_col}_grouped"] = df[cat_col].where(df[cat_col].isin(keep), other="прочее")
print("\nНовые значения и их размеры:")
print(df[f"{cat_col}_grouped"].value_counts().to_string())

Уникальных значений по кандидатам: {'grade': 10, 'bedrooms': 12, 'bathrooms': 22, 'floors': 6, 'view': 5, 'waterfront': 2, 'y_clf': 2}

Выбранный столбец: bathrooms, уникальных значений: 22
Десять самых частых значений покрывают 97.67% данных
Значений, встретившихся ровно один раз: 4

Категорий, покрывающих 90% данных: 8
Значения: [2.5, 1.0, 1.75, 2.0, 2.25, 1.5, 2.75, np.float64(3.0)]

Новые значения и их размеры:
bathrooms_grouped
2.5       538
1.0       527
1.75      373
2.0       255
2.25      227
1.5       193
прочее    185
2.75      119
3.0        68


**Ответ.** _Здесь письменный ответ на вопрос пункта._

> Столбец bathrooms имеет 22 уникальных значения. Десять самых частых покрывают 97.67% данных, но 4 значения встретились ровно один раз. Категорий, покрывающих 90%, - 8 (от 1.0 до 3.0), остальные в «прочее».

> Если кодировать как есть, получилось бы 22 столбца. Это плохо, потому что в столбцах для редких значений (7.5, 6.75 и т.д.) лежит по 1–2 дома - модель просто запомнит их и переобучится. Плюс bathrooms - это числовая шкала, и кодирование ломает её порядок. Лучше оставить её числом или сгруппировать, как мы и сделали.

## ○ Пункт 8. Паспорт данных

<sub>дополнительный пункт · 1 балл</sub>

In [9]:
# Пункт 8: код


**Ответ.** _Здесь письменный ответ на вопрос пункта._

> Три главные проблемы:

> 1) Невозможные значения в bedrooms: три дома имеют -2 спальни, что физически невозможно. Ошибка охватывает лишь 0.12% строк, но она «заражает» средние и стандартные отклонения, из-за которых правило трёх сигм потом метит не то, что нужно.

> 2) Пропуски в sqft_basement: 290 строк (11.67%) не имеют значения, и, как показал пункт 2, они не случайны: дома без подвала в среднем старее (1960 против 1968), меньше по площади sqft_above (1598 против 1708) и чаще имеют y_clf = 1 (77.6% против 29.6%).

> 3) Длинный хвост редких категорий в bathrooms: 22 уникальных значения, из которых 4 встречаются ровно один раз, а десять самых частых покрывают 97.67% - кодирование дало бы 22 столбца, где часть заполнена 1–2 наблюдениями.

> Что надо сделать до моделирования:

> 1) Исправить bedrooms = -2 - заменить, потому что остальные признаки в этих строках нормальные, а терять данные из-за одной ячейки неразумно.

> 2) Удалить один из двух частичных дубликатов по [price, sqft_living, sqft_lot, yr_built, lat, long], иначе один и тот же дом учитывается дважды.

> 3) Заполнить sqft_basement нулём, а не средним, потому что пропуск означает «подвала нет», а не «неизвестно».

> 4) Сгруппировать bathrooms в 8 категорий плюс «прочее», чтобы избежать переобучения на редких значениях.

> 5) Оставить price_per_sqft — корреляция с исходной ценой всего +0.38, признак не дублирует её и снижает коэффициент вариации с 0.54 до 0.33.

> Чего эти данные не покажут.

> Главный содержательный вопрос в недвижимости: «Стоит ли покупать этот дом за эту цену?», на который эти данные ответить не могут, потому что в них нет информации о состоянии и комфортности дома. Нет данных о школах, преступности, пробках, планах застройки - то есть о том, что на самом деле определяет цену. Именно поэтому две одинаковые по признакам квартиры могут стоить совершенно по-разному, что модель не обоснует.



## ● Пункт 9. Воспроизводимость

<sub>обязательный пункт · 1 балл</sub>

In [10]:
import pandas as pd
import numpy as np
import sys

print(f"pandas: {pd.__version__}")
print(f"numpy:  {np.__version__}")
print(f"Python: {sys.version.split()[0]}")

pandas: 2.2.3
numpy:  2.1.3
Python: 3.13.16


**Ответ.** _Здесь письменный ответ на вопрос пункта._

> После перезапуска ядра и выполнения всего ноутбука сверху вниз числа не абсолютно не изменились изменились, напиример:
- пропуски в sqft_basement: было 290 из 2485 (11.67%) — стало 290 (11.67%);
- среднее price для view = 0: было 482 718.42 — стало 482 718.42;
- строк с bedrooms < 0: было 3 — стало 3.

> Я уверена в совпадении, потому что в коде нет операций со случайностью и других вызовов, которые могли бы дать разные числа от запуска к запуску. Все вычисления детерминированы и зависят только от содержимого файла "data.csv".

Версии зафиксированы:
- pandas: 2.2.3
- numpy: 2.1.3
- Python: 3.13.16

> Абсолютных путей файл не содержит.

---

## Перед сдачей

- [ ] Kernel → Restart & Run All прошёл без ошибок
- [ ] выводы ячеек сохранены в файле
- [ ] у каждого пункта есть письменный ответ
- [ ] на каждом графике подписаны обе оси
- [ ] в первой ячейке указаны фамилия, группа и номер варианта
- [ ] Контест по этому заданию принят — без него работа не рассматривается